# 🌲 Análise de Volume e Biometria de Eucalipto
---
Este notebook é um ambiente interativo para a consolidação e análise das métricas dendrométricas geradas pelo nosso pipeline de processamento LiDAR.

### Objetivos:
1. Analisar as saídas das redes neurais de segmentação semântica e de instâncias (FF3D e TreeIso).
2. Processar e validar as reconstruções topológicas de RayExtract a partir de QSMs (`.txt` e `.csv`).
3. Gerar tabelas e visualizações consolidadas de DAP (Diâmetro à Altura do Peito) e Volume Total/Comercial.

### Como usar:
* Altere a variável `PIPELINE_KIND` para alternar entre os modelos de predição via nuvem de pontos ou via estrutura de árvore (RayExtract).
* Altere `EVALUATION_KIND` para `'single'` (teste rápido de um método) ou `'all'` (benchmarking completo).

In [1]:
# =====================================================================
# 1. IMPORTAÇÕES E SETUP DE AMBIENTE
# =====================================================================
import re
import sys
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# Configuração visual para os gráficos
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams['figure.figsize'] = (10, 6)

# Configuração do Path do Projeto
PROJECT_ROOT = Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Importações dos módulos do pipeline
from process_commercial_volume import (
    DEFAULT_DBH_METHODS,
    DEFAULT_VOLUME_METHODS,
    compare_method_combinations,
)
from eucalipto.io import load_ply, save_ply_with_fields
from eucalipto.rayextract_processor import process_rayextract_file

print("Ambiente configurado com sucesso!")

Ambiente configurado com sucesso!


/home/matheuspimenta/miniconda3/envs/euc/lib/python3.12/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


### 2. Configurações Globais e Parâmetros
Defina aqui qual pipeline você quer rodar e os parâmetros biológicos do inventário.

In [2]:
# =====================================================================
# 2. CONFIGURAÇÕES
# =====================================================================
# Opções: 'point_cloud' (FF3D/TreeISO) ou 'rayextract' (QSM)
PIPELINE_KIND = 'point_cloud'

# Opções: 'single' (Apenas os métodos abaixo) ou 'all' (Todos os métodos disponíveis)
EVALUATION_KIND = 'all'  
SINGLE_DBH_METHOD = 'ensemble'
SINGLE_VOLUME_METHOD = 'axis_profile'

# Constante Biológica
WOOD_DENSITY_KG_M3 = 900.0
POINT_CLOUD_CROP_FRACTION = 0.25

# ---------------------------------------------------------------------
# PARÂMETROS PARA 'point_cloud' (FF3D / TreeIso)
# ---------------------------------------------------------------------
POINT_CLOUD_PATH = Path('/home/matheuspimenta/Jobs/Eucalipto/FF3D_inference/ff3d_forestsens/FF3D_oracle/bucket_out_folder/diego_talhao_62fixedname_round2.ply')
POINT_CLOUD_TREE_ID_CANDIDATES = ['tree_id', 'instance_pred', 'treeID', 'final_segs']
POINT_CLOUD_TRUNK_CANDIDATES = ['trunk_leaf_label', 'semantic_seg', 'semantic_pred', 'leafwood_pred']
POINT_CLOUD_TRUNK_LABEL = 1

# ---------------------------------------------------------------------
# PARÂMETROS PARA 'rayextract'
# ---------------------------------------------------------------------
RAYEXTRACT_TREEFILES = {
    'plot_1': Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/canoa/tests/62/rayextract_full/rayextract_work/diego_talhao_62_raycloud_trees.txt'),
    # 'plot_2': Path('/home/matheuspimenta/Jobs/Eucalipto/drive/outputs/ray/CULS_CULS_plot_2_annotated_test_raycloud_trees.txt'),
    # 'plot_3': Path('/home/matheuspimenta/Jobs/Eucalipto/drive/outputs/ray/CULS_CULS_plot_3_annotated_val_raycloud_trees.txt'),
}

# Meshes da reconstrução 3D do RayExtract para comparação com o .txt.
# Informe uma lista de arquivos .ply por plot (ex.: tree_mesh.ply por árvore).
RAYEXTRACT_MESH_FILES = {
    'plot_1': [
        Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/canoa/tests/62/rayextract_full/rayextract_work/diego_talhao_62_raycloud_trees_mesh.ply'),
    #     Path('/home/matheuspimenta/Jobs/Eucalipto/projeto_eucalipto/results/.../tree_001_mesh.ply'),
    ],
}

print(f"Pipeline Selecionado: {PIPELINE_KIND.upper()}")
print(f"Modo de Avaliação: {EVALUATION_KIND.upper()}")

Pipeline Selecionado: POINT_CLOUD
Modo de Avaliação: ALL


In [3]:
# =====================================================================
# 3. FUNÇÃO CORE DE PROCESSAMENTO (Point Cloud)
# =====================================================================
def crop_point_cloud_lower_left_quadrant(cloud_path, output_dir, crop_fraction=POINT_CLOUD_CROP_FRACTION):
    """Recorta a nuvem no quadrante inferior esquerdo do bounding box."""
    points, extras = load_ply(str(cloud_path))
    if points.size == 0:
        raise ValueError(f'Point cloud vazia: {cloud_path}')

    x_min, y_min = points[:, 0].min(), points[:, 1].min()
    x_max, y_max = points[:, 0].max(), points[:, 1].max()
    x_cut = x_min + (x_max - x_min) * float(crop_fraction)
    y_cut = y_min + (y_max - y_min) * float(crop_fraction)

    crop_mask = (points[:, 0] <= x_cut) & (points[:, 1] <= y_cut)
    cropped_points = points[crop_mask]
    cropped_extras = {name: values[crop_mask] for name, values in extras.items()}

    if cropped_points.size == 0:
        raise ValueError(
            f'Cropping removed all points from {cloud_path.name}. Check crop_fraction or cloud extent.'
        )

    cropped_path = Path(output_dir) / f'{Path(cloud_path).stem}_crop_{int(crop_fraction * 100):02d}.ply'
    save_ply_with_fields(str(cropped_path), cropped_points, cropped_extras)
    return cropped_path, int(cropped_points.shape[0]), int(points.shape[0])

def run_point_cloud_evaluation(
    cloud_path,
    dbh_methods,
    volume_methods,
    trunk_label=POINT_CLOUD_TRUNK_LABEL,
    wood_density_kg_m3=WOOD_DENSITY_KG_M3,
    tree_id_candidates=None,
    trunk_candidates=None,
):
    """Encapsula a lógica de extração de métricas de nuvens de pontos segmentadas."""
    tree_id_candidates = tree_id_candidates or POINT_CLOUD_TREE_ID_CANDIDATES
    trunk_candidates = trunk_candidates or POINT_CLOUD_TRUNK_CANDIDATES

    # Executa os algoritmos de extração do pipeline
    dbh_df, combo_df = compare_method_combinations(
        str(cloud_path),
        tree_id_candidates=tree_id_candidates,
        trunk_leaf_candidates=trunk_candidates,
        trunk_label=trunk_label,
        wood_density_kg_m3=wood_density_kg_m3,
        dbh_methods=dbh_methods,
        volume_methods_list=volume_methods,
    )

    # Limpeza de dados nulos/inválidos (-1)
    if not dbh_df.empty and 'tree_id' in dbh_df.columns:
        dbh_df = dbh_df[dbh_df['tree_id'].fillna(-1).astype(int) >= 0].copy()
    if not combo_df.empty and 'tree_id' in combo_df.columns:
        combo_df = combo_df[combo_df['tree_id'].fillna(-1).astype(int) >= 0].copy()

    return dbh_df, combo_df

In [4]:
# =====================================================================
# 4. EXECUÇÃO DO PIPELINE: NUVEM DE PONTOS (FF3D / TreeIso)
# =====================================================================
if PIPELINE_KIND == 'point_cloud':
    if not POINT_CLOUD_PATH.exists():
        raise FileNotFoundError(f'ERRO: Arquivo de nuvem não encontrado em: {POINT_CLOUD_PATH}')

    print(f"Iniciando processamento da nuvem: {POINT_CLOUD_PATH.name}...")
    
    # Define a lista de métodos baseada no tipo de avaliação escolhida
    if EVALUATION_KIND == 'single':
        selected_dbh_methods = [SINGLE_DBH_METHOD]
        selected_volume_methods = [SINGLE_VOLUME_METHOD]
        print(f"Rodando análise rápida -> DAP: '{SINGLE_DBH_METHOD}' | Vol: '{SINGLE_VOLUME_METHOD}'")
        
    elif EVALUATION_KIND == 'all':
        selected_dbh_methods = DEFAULT_DBH_METHODS
        selected_volume_methods = DEFAULT_VOLUME_METHODS
        print(f"🌎 Rodando benchmarking completo para todos os métodos.")
    else:
        raise ValueError("ERRO: EVALUATION_KIND deve ser 'single' ou 'all'")

    # Recorta a nuvem antes da análise para reduzir o custo computacional
    with tempfile.TemporaryDirectory(prefix='eucalipto_crop_') as crop_dir:
        cropped_cloud_path, cropped_n_points, original_n_points = crop_point_cloud_lower_left_quadrant(
            POINT_CLOUD_PATH,
            crop_dir,
            crop_fraction=POINT_CLOUD_CROP_FRACTION,
        )
        print(
            f"Nuvem recortada para {cropped_n_points}/{original_n_points} pontos ",
            f"({cropped_n_points / original_n_points:.1%}) no quadrante inferior esquerdo."
        )

        # Executa a função core e obtém os DataFrames
        dbh_df, volume_df = run_point_cloud_evaluation(
            cropped_cloud_path,
            dbh_methods=selected_dbh_methods,
            volume_methods=selected_volume_methods,
        )
    print("Processamento concluído com sucesso!")
    print(f"Árvores identificadas para DAP: {len(dbh_df)} | Combinações volumétricas: {len(volume_df)}")

Iniciando processamento da nuvem: diego_talhao_62fixedname_round2.ply...
🌎 Rodando benchmarking completo para todos os métodos.
Nuvem recortada para 217325/3783057 pontos  (5.7%) no quadrante inferior esquerdo.
Processamento concluído com sucesso!
Árvores identificadas para DAP: 177 | Combinações volumétricas: 885


### Resultados da Extração Direta da Nuvem (Point Cloud)
Os DataFrames abaixo consolidam os resultados do *fitting* por indivíduo arbóreo e o sumário estatístico do método de volumetria escolhido.

In [5]:
# =====================================================================
# 5. VISUALIZAÇÃO DE DADOS (Point Cloud)
# =====================================================================
if PIPELINE_KIND == 'point_cloud':
    
    # 5.1. Resultados Individuais (Árvore a Árvore)
    display(Markdown("#### **Resultados de DAP por Árvore**"))
    display(dbh_df.sort_values(['tree_id', 'method']).reset_index(drop=True))

    display(Markdown("#### **Resultados Volumétricos por Árvore e Combinação de Método**"))
    display(volume_df.sort_values(['tree_id', 'dbh_method', 'volume_method']).reset_index(drop=True))

    # 5.2. Sumário Estatístico (Apenas se o volume_df não estiver vazio)
    if not volume_df.empty:
        display(Markdown("#### **Sumário Global por Estratégia de Volume**"))
        summary = (
            volume_df.groupby('volume_method', dropna=False)
            .agg(
                Qtd_Árvores=('tree_id', 'nunique'),
                Volume_Médio_m3=('volume_m3', 'mean'),
                Volume_Mediano_m3=('volume_m3', 'median'),
                Vol_Comercial_Médio_m3=('commercial_volume_m3', 'mean'),
                Massa_Média_kg=('mass_kg', 'mean'),
            )
            .reset_index()
            .sort_values('volume_method')
        )
        # Exibe com fundo gradiente para facilitar a comparação
        display(summary.style.background_gradient(subset=['Volume_Médio_m3'], cmap='viridis'))

        if EVALUATION_KIND == 'all':
            display(Markdown("#### **Comparativo Cruzado: Método de DAP x Método de Volume**"))
            comparison = (
                volume_df.groupby(['dbh_method', 'volume_method'], dropna=False)
                .agg(
                    Qtd_Árvores=('tree_id', 'nunique'),
                    Volume_Médio_m3=('volume_m3', 'mean'),
                    Vol_Comercial_Médio_m3=('commercial_volume_m3', 'mean'),
                    Massa_Média_kg=('mass_kg', 'mean'),
                )
                .reset_index()
                .sort_values(['dbh_method', 'volume_method'])
            )
            display(comparison.style.background_gradient(subset=['Volume_Médio_m3'], cmap='plasma'))

#### **Resultados de DAP por Árvore**

,cloud,tree_id,method,dbh_cm,status,n_values
0,diego_talhao_62fixedname_round2_crop_25,0,ensemble,NaN,none,0
1,diego_talhao_62fixedname_round2_crop_25,0,ls,NaN,none,0
2,diego_talhao_62fixedname_round2_crop_25,0,single_ransac,NaN,none,0
3,diego_talhao_62fixedname_round2_crop_25,1,ensemble,31.311255,ok,1
4,diego_talhao_62fixedname_round2_crop_25,1,ls,28.737394,ok,0
...,...,...,...,...,...,...
172,diego_talhao_62fixedname_round2_crop_25,829,ls,NaN,none,0
173,diego_talhao_62fixedname_round2_crop_25,829,single_ransac,NaN,none,0
174,diego_talhao_62fixedname_round2_crop_25,855,ensemble,56.871758,ok,1
175,diego_talhao_62fixedname_round2_crop_25,855,ls,79.582553,ok,0


#### **Resultados Volumétricos por Árvore e Combinação de Método**

,cloud,tree_id,dbh_method,volume_method,dbh_cm,height_m,cbc_rel_m,cbc_abs_m,volume_m3,mass_kg,commercial_volume_m3,commercial_mass_kg,status,n_trunk_pts,n_commercial_pts
0,diego_talhao_62fixedname_round2_crop_25,0,ensemble,axis_profile,NaN,9.5897,9.589700,13.594400,1.918834,1726.950669,1.918834,1726.950669,ok,490,490
1,diego_talhao_62fixedname_round2_crop_25,0,ensemble,cylinder,NaN,9.5897,9.589700,13.594400,NaN,NaN,NaN,NaN,error: dbh_cm or CBC is missing for cylinder v...,490,490
2,diego_talhao_62fixedname_round2_crop_25,0,ensemble,frustum,NaN,9.5897,9.589700,13.594400,1.961726,1765.553344,1.360554,1224.499036,ok,490,490
3,diego_talhao_62fixedname_round2_crop_25,0,ensemble,taper,NaN,9.5897,9.589700,13.594400,4.981100,4482.989683,2.065937,1859.343466,ok,490,490
4,diego_talhao_62fixedname_round2_crop_25,0,ensemble,voxel,NaN,9.5897,9.589700,13.594400,0.059875,53.887500,0.059875,53.887500,ok,490,490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
880,diego_talhao_62fixedname_round2_crop_25,855,single_ransac,axis_profile,NaN,1.4408,1.426392,5.190092,0.076995,69.295851,0.067175,60.457119,ok,64,63
881,diego_talhao_62fixedname_round2_crop_25,855,single_ransac,cylinder,NaN,1.4408,1.426392,5.190092,NaN,NaN,NaN,NaN,error: dbh_cm or CBC is missing for cylinder v...,64,63
882,diego_talhao_62fixedname_round2_crop_25,855,single_ransac,frustum,NaN,1.4408,1.426392,5.190092,1.975934,1778.340360,0.315944,284.349706,ok,64,63
883,diego_talhao_62fixedname_round2_crop_25,855,single_ransac,taper,NaN,1.4408,1.426392,5.190092,0.997460,897.714179,0.405056,364.550493,ok,64,63


#### **Sumário Global por Estratégia de Volume**

,volume_method,Qtd_Árvores,Volume_Médio_m3,Volume_Mediano_m3,Vol_Comercial_Médio_m3,Massa_Média_kg
0,axis_profile,59,5.647020,2.403200,7.146972,5082.317817
1,cylinder,59,5.608633,2.611428,5.459321,5047.769258
2,frustum,59,4.589222,2.109447,5.433333,4130.299915
3,taper,59,468.786729,3.582101,589.081755,421908.055895
4,voxel,59,0.044291,0.040500,0.042526,39.861853


#### **Comparativo Cruzado: Método de DAP x Método de Volume**

,dbh_method,volume_method,Qtd_Árvores,Volume_Médio_m3,Vol_Comercial_Médio_m3,Massa_Média_kg
0,ensemble,axis_profile,59,5.647020,7.146972,5082.317817
1,ensemble,cylinder,59,5.054388,4.935230,4548.949254
2,ensemble,frustum,59,4.722063,5.056684,4249.856671
3,ensemble,taper,59,134.220837,1315.427504,120798.753614
4,ensemble,voxel,59,0.044291,0.042526,39.861853
5,ls,axis_profile,59,5.647020,7.146972,5082.317817
6,ls,cylinder,59,6.363086,6.209185,5726.777129
7,ls,frustum,59,4.789806,5.432071,4310.825189
8,ls,taper,59,4.357102,452.583892,3921.392089
9,ls,voxel,59,0.044291,0.042526,39.861853


In [7]:
volume_df.dropna(subset=['volume_m3'], inplace=False).shape

(497, 15)

In [ ]:
# =====================================================================
# 6. EXECUÇÃO DO PIPELINE: RAYEXTRACT (QSMs + Meshes - Otimizado)
# =====================================================================
import re
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

if PIPELINE_KIND == 'rayextract':
    def _infer_tree_id_from_name(mesh_path: Path, fallback_id: int) -> int:
        matches = re.findall(r'\d+', mesh_path.stem)
        return int(matches[-1]) if matches else int(fallback_id)

    def process_single_mesh(mesh_path, wood_density_kg_m3, fallback_id):
        """Processa um único arquivo de malha de forma isolada (Ideal para paralelismo)"""
        import trimesh
        source_tree_hint = _infer_tree_id_from_name(mesh_path, fallback_id)
        
        try:
            # force='mesh' garante que seja um único objeto, ignorando hierarquias visuais da cena
            mesh = trimesh.load(str(mesh_path), force='mesh')
            
            # --- OTIMIZAÇÃO DE PERFORMANCE AQUI ---
            # Em vez de mesh.split(), calculamos as métricas globais da malha consolidada.
            z_min = float(mesh.bounds[0][2])
            z_max = float(mesh.bounds[1][2])
            height_m = float(max(0.0, z_max - z_min))

            # Tenta pegar o volume direto da malha. Se a malha estiver 'furada', ele retorna 0 ou erro.
            volume_source = 'mesh_global'
            volume_m3 = float(abs(mesh.volume)) if mesh.volume is not None else np.nan
            
            # Fallback rápido: Convex Hull da nuvem de vértices
            if not np.isfinite(volume_m3) or volume_m3 <= 0:
                hull = mesh.convex_hull
                volume_m3 = float(abs(hull.volume))
                volume_source = 'convex_hull_global'

            mass_kg = float(volume_m3 * wood_density_kg_m3) if np.isfinite(volume_m3) else np.nan

            return {
                'tree_id': fallback_id,
                'source_tree_hint': source_tree_hint,
                'mesh_file': mesh_path.name,
                'mesh_component_id': 0, # Tratado como componente único
                'component_count': 1,
                'volume_m3': volume_m3,
                'mass_kg': mass_kg,
                'height_m': height_m,
                'volume_source': volume_source,
                'status': 'success',
            }
        except Exception as exc:
            return {
                'tree_id': fallback_id,
                'source_tree_hint': source_tree_hint,
                'mesh_file': mesh_path.name,
                'mesh_component_id': None,
                'component_count': np.nan,
                'volume_m3': np.nan,
                'mass_kg': np.nan,
                'height_m': np.nan,
                'volume_source': 'error',
                'status': f'error: {exc}',
            }

    def process_rayextract_mesh_files_fast(mesh_paths, wood_density_kg_m3=WOOD_DENSITY_KG_M3):
        """Gerencia a extração usando pool de processos para não travar o notebook."""
        try:
            import trimesh
        except ImportError:
            print("⚠️ [AVISO] Pacote 'trimesh' não instalado. Pulando análise por mesh (.ply).")
            return pd.DataFrame()

        rows = []
        valid_paths = sorted({Path(p) for p in mesh_paths if Path(p).exists()})
        
        print(f"⏳ Processando {len(valid_paths)} meshes de forma otimizada...")
        start_time = time.time()
        
        # Uso de paralelismo para acelerar leitura de I/O pesado
        with ProcessPoolExecutor() as executor:
            futures = [
                executor.submit(process_single_mesh, path, wood_density_kg_m3, idx)
                for idx, path in enumerate(valid_paths)
            ]
            for future in as_completed(futures):
                rows.append(future.result())

        elapsed = time.time() - start_time
        print(f"⚡ Extração de Meshes concluída em {elapsed:.2f} segundos!")
        
        out = pd.DataFrame(rows)
        if not out.empty:
            out = out.sort_values(['mesh_file']).reset_index(drop=True)
        return out

    # =========================================================
    # Lógica Principal do Pipeline
    # =========================================================
    ray_txt_results = {}
    ray_mesh_results = {}

    print("🚀 Iniciando pipeline RayExtract (.txt vs .ply)...")

    for plot_name, txt_path in RAYEXTRACT_TREEFILES.items():
        print(f"\n================== {plot_name.upper()} ==================")

        # A) Estrutura cilíndrica (.txt)
        if txt_path.exists():
            print(f"📄 [TXT] Processando: {txt_path.name}")
            df_ray_txt = process_rayextract_file(str(txt_path), wood_density_kg_m3=WOOD_DENSITY_KG_M3)
            df_ray_txt.insert(0, 'plot', plot_name)
            ray_txt_results[plot_name] = df_ray_txt
        else:
            print(f"⚠️ [TXT] Arquivo não encontrado: {txt_path}")

        # B) Reconstrução geométrica (.ply) otimizada
        mesh_files_cfg = RAYEXTRACT_MESH_CSVS.get(plot_name, []) # Correção da variável que chama os meshes
        mesh_files = [Path(p) for p in mesh_files_cfg] if isinstance(mesh_files_cfg, list) else []

        if mesh_files:
            df_ray_mesh = process_rayextract_mesh_files_fast(mesh_files, wood_density_kg_m3=WOOD_DENSITY_KG_M3)
            if not df_ray_mesh.empty:
                df_ray_mesh.insert(0, 'plot', plot_name)
                ray_mesh_results[plot_name] = df_ray_mesh
        else:
            print(f"⚠️ [MESH] Nenhum arquivo .ply configurado para {plot_name}.")

    # =========================================================
    # CONSOLIDAÇÃO VISUAL
    # =========================================================
    ray_txt_all = pd.concat(ray_txt_results.values(), ignore_index=True) if ray_txt_results else pd.DataFrame()
    ray_mesh_all = pd.concat(ray_mesh_results.values(), ignore_index=True) if ray_mesh_results else pd.DataFrame()
    
    if (not ray_txt_all.empty) and (not ray_mesh_all.empty):
        txt_cmp = ray_txt_all.groupby('plot', dropna=False).agg(
            txt_vol_m3=('volume_m3', 'sum'), txt_qtd=('tree_id', 'nunique')).reset_index()

        mesh_cmp = ray_mesh_all.groupby('plot', dropna=False).agg(
            mesh_vol_m3=('volume_m3', 'sum'), mesh_qtd=('tree_id', 'nunique')).reset_index()

        comparison = txt_cmp.merge(mesh_cmp, on='plot', how='outer')
        comparison['Erro_Relativo_%'] = ((comparison['mesh_vol_m3'] - comparison['txt_vol_m3']) / comparison['txt_vol_m3']) * 100.0

        display(Markdown("#### **📊 Análise de Discrepância: Estrutura (.txt) vs Malha 3D (.ply)**"))
        display(comparison.style.background_gradient(subset=['Erro_Relativo_%'], cmap='coolwarm').format(precision=2))

Iniciando pipeline RayExtract com duas abordagens: .txt e .ply...

================== PLOT_1 ==================
[TXT] Processando: diego_talhao_62_raycloud_trees.txt


#### **Preview - Métricas via estrutura .txt**

,plot,tree_id,dbh_cm,volume_m3,mass_kg,height_m
0,plot_1,0,3.68,50640.519175,4.557647e+07,30.4164
1,plot_1,1,3.90,7.336156,6.602540e+03,27.5850
2,plot_1,2,25.96,15.754448,1.417900e+04,28.3197
3,plot_1,3,4.26,2.730030,2.457027e+03,27.7416
4,plot_1,4,4.22,4.756383,4.280745e+03,27.6914


[MESH] 1 arquivo(s) configurado(s) para plot_1.


KeyboardInterrupt: 